## Analisi Strutturale della Rete Terroristica

Questo codice si focalizza sull'Analisi Strutturale di Rete (SNA) di un dataset relazionale che mappa le interconnessioni all'interno di una o più organizzazioni terroristiche.

L'_obiettivo_ è trasformare i dati grezzi in un grafo analizzabile e utilizzare gli attributi di nodo per segmentare e interpretare i risultati.

***Dati Sorgente***

I dati sono composti da due file principali:

- _TerroristRel.edges_: File di testo che definisce gli archi del grafo. Ogni riga rappresenta un legame tra due nodi distinti, dove ciascun nodo identifica univocamente un individuo (agente).

- _TerroristRel.node_labels_ : File che assegna un attributo categoriale (etichetta) a ogni nodo, con valori discreti 0 o 1. Sebbene la natura esatta dell'attributo sia ambigua (es. ruolo, status, fazione), questa etichetta sarà fondamentale per l'analisi comparativa, la segmentazione e la colorazione del grafo (es. studio dell'omofilia).

**Prossimi Passi**

Dopo l'importazione e l'assegnazione degli attributi, si procederà con l'analisi descrittiva (Densità, Grado Medio) e la visualizzazione del grafo.

In [ ]:
# Installazione dei pacchetti necessari
%pip install -r requirements.txt


import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

## Preparazione dei Dati e Assegnazione degli Attributi di Rete

Questo blocco di codice si occupa di caricare il grafo e associare gli attributi categoriali (etichette) ai nodi, preparando la rete per l'analisi e la visualizzazione.

_Passaggi Chiave_:
- Caricamento del Grafo: Il codice importa gli archi dal file TerroristRel.edges (assumendo che gli archi siano separati da virgole e che gli ID dei nodi siano numeri interi), creando l'oggetto grafo base G in NetworkX.

- Lettura delle Etichette: Viene aperto e letto il file TerroristRel.node_labels. Per ogni riga, l'ID del nodo e la sua etichetta (1 o 2) vengono estratti e salvati nel dizionario Python node_labels.

- Assegnazione dell'Attributo: La funzione cruciale nx.set_node_attributes() associa il dizionario node_labels all'oggetto grafo G, memorizzando i valori 1 e 2 sotto il nome di attributo 'label' per ogni nodo.

        Questo passaggio è essenziale per l'analisi comparativa e la colorazione.

- Verifica: Infine, vengono stampati i primi 5 nodi con le loro etichette per una rapida verifica del corretto caricamento dei dati.

In [ ]:
G = nx.read_edgelist('TerroristRel.edges', delimiter=',', nodetype=int) 

# Assegnazione etichette
node_labels = {}
with open('TerroristRel.node_labels', 'r') as f:
    for line in f:
        parts = line.strip().split(',') # Split riga
        if len(parts) == 2:
            # Associa l'ID del nodo (chiave) all'Etichetta (valore 0 o 1)
            node_id = int(parts[0])
            label = int(parts[1])
            node_labels[node_id] = label

nx.set_node_attributes(G, node_labels, 'label') # Aggiungiamo l'etichetta

# Stampa etichette
items = list(node_labels.items())
head = items[:5]
print(head)

Il grafo G è ora pronto per calcolare metriche e caratteristiche del grafom centralità separate per l'etichetta (1 o 2) e per generare la visualizzazione 3D bicolore.

In [ ]:
print('Massimo grado di un nodo:', max(dict(G.degree()).values()))
print('Minimo grado di un nodo:', min(dict(G.degree()).values()))
print('Grado medio dei nodi:',sum(dict(G.degree()).values()) / G.number_of_nodes())

# Analisi Descrittiva della Topologia di Rete

Lo script calcola e stampa le principali metriche topologiche che descrivono la dimensione, la coesione e la connettività del grafo, gestendo in modo condizionale le metriche che richiedono la connettività della rete.

1. _Metriche Base_

**n_nodes** e **n_edges**: Calcolano il numero totale di individui (nodi) e il numero totale di relazioni (archi) nella rete.

**density**: Calcola la densità del grafo, ovvero il rapporto tra gli archi esistenti e gli archi massimi possibili. Questo indica quanto è connessa la rete nel suo complesso.

**connected**: Verifica se il grafo è connesso, cioè se è possibile raggiungere ogni nodo da ogni altro nodo.

2. _Metriche di Distanza (Condizionali)_

Queste metriche sono calcolate solo se il grafo risulta essere connesso (if connected:), poiché non hanno significato in un grafo disconnesso (formato da più componenti isolate).

**diameter**: Calcola il diametro del grafo, ovvero la distanza massima (il percorso più lungo) tra due nodi qualsiasi della rete.

**radius**: Calcola il raggio del grafo, che è la distanza minima tra il centro della rete e il nodo più distante da esso.

**periphery**: Identifica i nodi che si trovano sulla periferia della rete (i nodi la cui massima distanza da qualsiasi altro nodo è pari al diametro).

3. _Misura della Coesione Locale_

**avg_clustering**: Calcola il coefficiente di clustering medio. Questa metrica indica la probabilità media che due vicini di un nodo siano a loro volta collegati tra loro, misurando la tendenza della rete a formare cluster o cricche locali.

In [ ]:
n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()
density = nx.density(G)
connected = nx.is_connected(G)
avg_clustering = nx.average_clustering(G)

if connected:
    radius = nx.radius(G)
    diameter = nx.diameter(G)
    periphery = nx.periphery(G)
else:
    radius = None
    diameter = None
    periphery = None

print(f"Number of nodes: {n_nodes}")
print(f"Number of edges: {n_edges}")
print(f"Density: {density}")
print(f"Connected: {connected}")
    
if connected:
    print(f"Radius: {radius}")
    print(f"Diameter: {diameter}")
    print(f"Periphery: {periphery}")
else:
    print("Radius, diameter and periphery are not defined for disconnected graphs.")
    
print(f"Average clustering coefficient: {avg_clustering}")

## Analisi Qualitativa e Strutturale della Rete Terroristica

Questa sezione si concentra sull'analisi qualitativa della struttura di rete attraverso l'applicazione di diversi algoritmi di layout. L'obiettivo è estrarre caratteristiche visive del grafo che possano essere direttamente ricondotte alla coesione, alla gerarchia e alla natura delle interconnessioni all'interno dell'organizzazione terroristica.

Sono stati utilizzati algoritmi di layout basati sulla forza, topologici e casuali, esplorandone le dimensioni 2D e 3D per ottenere una visione completa:

# I. Layout Basati sulla Forza (Force-Directed)

Questi layout ottimizzano la visualizzazione posizionando i nodi vicini se sono fortemente connessi (simulando attrazione) e lontani se non lo sono (simulando repulsione).

_Kamada-Kawai (2D/3D) e Spring Layout (2D/3D)_:

**Obiettivo**: Rivelare la struttura a cluster (comunità) e i nodi centrali (hub).

**Deduzione Organizzativa**: I raggruppamenti densi che emergono (specialmente in 3D) rappresentano individui con forti e rapidi canali di comunicazione. La visualizzazione 3D è cruciale per risolvere le sovrapposizioni e confermare l'esistenza di nodi che fungono da ponti critici (bottleneck) tra i sottogruppi (evidenziati dalla colorazione per etichetta 1 vs 2).

# II. Layout Topologici e Circolari

Questi layout si basano sulla struttura predefinita per l'analisi.

_Circular Layout (2D/3D)_:

**Obiettivo**: Visualizzare la distribuzione complessiva delle connessioni.

**Deduzione Organizzativa**: Permette una chiara distinzione visiva tra il volume di connessioni intra-gruppo (all'interno della stessa etichetta, 1 o 2) e inter-gruppo (tra etichette diverse), quantificando l'omofilia e la necessità di coordinamento tra le fazioni/ruoli.

_Spiral Layout (Solo 2D)_:

**Obiettivo**: Ordinare i nodi in base a una metrica (spesso il grado o la centralità) lungo una spirale, evidenziando il centro della rete.

# III. Layout Avanzati e Casuali

_ARF Layout (Attribute-aware force-directed - Solo 2D)_:

**Obiettivo**: Ottenere un layout che tiene conto esplicitamente dell'attributo durante il posizionamento, tendendo a raggruppare i nodi con lo stesso attributo.

**Deduzione Organizzativa**: Utile per confrontare la struttura di connessione con la struttura di appartenenza. Se il layout ARF assomiglia molto al layout Kamada-Kawai, suggerisce che l'appartenenza al gruppo è il fattore dominante nella formazione degli archi.

_Random Layout (Solo 2D)_:

**Obiettivo**: Fornire una baseline non strutturata.

**Deduzione Organizzativa**: Viene utilizzato come riferimento per dimostrare che i raggruppamenti visibili negli altri layout non sono casuali ma sono proprietà intrinseche della topologia di rete.

    **NOTA**: siccome il tempo per generare la totalità dei grafici 3D potrebbe richiedere qualche minuto, si consiglia di eseguire prima le visualizzazioni 2D di ognuno,poi eventualemente approfondire tramite il plot in 3 dimensioni.

In [ ]:
####--- 2D Kamada Kawai layout ---###

pos = nx.kamada_kawai_layout(G)


node_color_map = [G.nodes[node]['label'] for node in G.nodes()]

plt.figure(figsize=(30, 40))
nx.draw(
    G, 
    pos, 
    with_labels=False, 
    node_size=400, 
    node_color=node_color_map,  
    cmap=plt.cm.coolwarm,       
    font_size=5
)
plt.show()

In [ ]:
####--- 3D Kamada Kawai layout ---###

pos_3d = nx.kamada_kawai_layout(G, dim=3)

# --- 1. Preparazione degli Array di Coordinate e Colori ---

node_xyz = np.array([pos_3d[v] for v in G.nodes()])

# Estrarre la lista dei valori dell'etichetta ('label') e CONVERTIRE IN FLOAT
# Questa conversione è cruciale per forzare Plotly a trattarli come valori scalari.

node_colors = np.array([G.nodes[node]['label'] for node in G.nodes()], dtype=float)

# Estrazione delle coordinate per gli archi (percorsi x, y, z)
edge_x = []
edge_y = []
edge_z = []
for u, v in G.edges():
    x0, y0, z0 = pos_3d[u]
    x1, y1, z1 = pos_3d[v]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])
    edge_z.extend([z0, z1, None])

# --- 2. Preparazione del Testo Hover ---

node_text = []
for node in G.nodes():
    label_value = G.nodes[node]['label']
    degree_value = G.degree[node]
    node_text.append(f'ID Nodo: {node}<br>Etichetta: {label_value}<br>Grado: {degree_value}')

# --- 3. Creazione degli Oggetti Trace ---

edge_trace = go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    line=dict(width=0.5, color='#888'),
    hoverinfo='none',
    mode='lines')

vivid_colorscale = [
    [0.0, '#0000FF'],  # Rosso vivo
    [1.0, '#FF0000']   # Blu vivo
]

# Trace dei Nodi (Correzione del colore)
node_trace = go.Scatter3d(
    x=node_xyz[:, 0], y=node_xyz[:, 1], z=node_xyz[:, 2],
    text=node_text,
    mode='markers',
    hoverinfo='text',
    marker=dict(
        showscale=False,

        colorscale=vivid_colorscale, 
        
        # Assegnamo l'array di float
        color=node_colors,  
        
        # Forza i limiti per mappare 0 e 1 ai colori estremi della scala
        cmin=1.0,             
        cmax=2.0,             
        
        size=10,
        line=dict(width=1, color='DarkSlateGrey')
    )
)

# --- 4. Visualizzazione finale ---

fig = go.Figure(data=[edge_trace, node_trace],
                layout=go.Layout(
                    title='<br>Grafo 3D (Kamada-Kawai) dell\'organizzazione terroristica, colorata secondo i Label',
                    showlegend=False,
                    height=1000,
                    margin=dict(l=0, r=0, b=0, t=50),
                    scene=dict(
                        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        zaxis=dict(showgrid=False, zeroline=False, showticklabels=False))))

fig.show()

In [ ]:
####--- 2D Spring layout ---------###

pos = nx.spring_layout(G)


node_color_map = [G.nodes[node]['label'] for node in G.nodes()]

plt.figure(figsize=(20, 20))
nx.draw(
    G, 
    pos, 
    with_labels=True, 
    node_size=100, 
    node_color=node_color_map,  
    cmap=plt.cm.coolwarm,       
    font_size=5
)
plt.show()

In [ ]:
####--- 3D Spring layout ---------###

pos_3d = nx.spring_layout(G, dim=3)


# --- 1. Preparazione degli Array di Coordinate e Colori ---

node_xyz = np.array([pos_3d[v] for v in G.nodes()])

# Estrarre la lista dei valori dell'etichetta ('label') e CONVERTIRE IN FLOAT
# Questa conversione è cruciale per forzare Plotly a trattarli come valori scalari.
node_colors = np.array([G.nodes[node]['label'] for node in G.nodes()], dtype=float)

# Estrazione delle coordinate per gli archi (percorsi x, y, z)
edge_x = []
edge_y = []
edge_z = []
for u, v in G.edges():
    x0, y0, z0 = pos_3d[u]
    x1, y1, z1 = pos_3d[v]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])
    edge_z.extend([z0, z1, None])

# --- 2. Preparazione del Testo Hover ---

node_text = []
for node in G.nodes():
    label_value = G.nodes[node]['label']
    degree_value = G.degree[node]
    node_text.append(f'ID Nodo: {node}<br>Etichetta: {label_value}<br>Grado: {degree_value}')

# --- 3. Creazione degli Oggetti Trace ---

edge_trace = go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    line=dict(width=0.5, color='#888'),
    hoverinfo='none',
    mode='lines')

vivid_colorscale = [
    [0.0, '#0000FF'],  # Rosso vivo
    [1.0, '#FF0000']   # Blu vivo
]

# Trace dei Nodi (Correzione del colore)
node_trace = go.Scatter3d(
    x=node_xyz[:, 0], y=node_xyz[:, 1], z=node_xyz[:, 2],
    text=node_text,
    mode='markers',
    hoverinfo='text',
    marker=dict(
        showscale=False,

        colorscale=vivid_colorscale, 
        
        # Assegnamo l'array di float
        color=node_colors,  
        
        # Forza i limiti per mappare 0 e 1 ai colori estremi della scala
        cmin=1.0,             
        cmax=2.0,             
        
        size=10,
        line=dict(width=1, color='DarkSlateGrey')
    )
)

# --- 4. Visualizzazione finale ---

fig = go.Figure(data=[edge_trace, node_trace],
                layout=go.Layout(
                    title='<br>Grafo 3D (Spring) dell\'organizzazione terroristica, colorata secondo i Label',
                    showlegend=False,
                    height=1000,
                    margin=dict(l=0, r=0, b=0, t=50),
                    scene=dict(
                        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        zaxis=dict(showgrid=False, zeroline=False, showticklabels=False))))

fig.show()

In [ ]:
####--- 2D Circular layout -------###

pos = nx.circular_layout(G)


# Estraiamo la lista dei valori dell'attributo 'label'
node_color_map = [G.nodes[node]['label'] for node in G.nodes()]

plt.figure(figsize=(20, 20))
nx.draw(
    G, 
    pos, 
    with_labels=False, 
    node_size=100, 
    node_color=node_color_map,  # Usa la lista appena creata
    cmap=plt.cm.coolwarm,       # Colormap per distinguere i gruppi 0 e 1
    font_size=5
)
plt.show()

In [ ]:
####----3D Circular layout -------###

pos_3d = nx.circular_layout(G, dim=3)

# --- 1. Preparazione degli Array di Coordinate e Colori ---

node_xyz = np.array([pos_3d[v] for v in G.nodes()])

# Estrarre la lista dei valori dell'etichetta ('label') e CONVERTIRE IN FLOAT
# Questa conversione è cruciale per forzare Plotly a trattarli come valori scalari.
node_colors = np.array([G.nodes[node]['label'] for node in G.nodes()], dtype=float)

# Estrazione delle coordinate per gli archi (percorsi x, y, z)
edge_x = []
edge_y = []
edge_z = []
for u, v in G.edges():
    x0, y0, z0 = pos_3d[u]
    x1, y1, z1 = pos_3d[v]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])
    edge_z.extend([z0, z1, None])

# --- 2. Preparazione del Testo Hover ---

node_text = []
for node in G.nodes():
    label_value = G.nodes[node]['label']
    degree_value = G.degree[node]
    node_text.append(f'ID Nodo: {node}<br>Etichetta: {label_value}<br>Grado: {degree_value}')

# --- 3. Creazione degli Oggetti Trace ---

edge_trace = go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    line=dict(width=0.5, color='#888'),
    hoverinfo='none',
    mode='lines')

vivid_colorscale = [
    [0.0, '#0000FF'],  # Rosso vivo
    [1.0, '#FF0000']   # Blu vivo
]

# Trace dei Nodi (Correzione del colore)
node_trace = go.Scatter3d(
    x=node_xyz[:, 0], y=node_xyz[:, 1], z=node_xyz[:, 2],
    text=node_text,
    mode='markers',
    hoverinfo='text',
    marker=dict(
        showscale=False,

        colorscale=vivid_colorscale, 
        
        # Assegnamo l'array di float
        color=node_colors,  
        
        # Forza i limiti per mappare 0 e 1 ai colori estremi della scala
        cmin=1.0,             
        cmax=2.0,             
        
        size=10,
        line=dict(width=1, color='DarkSlateGrey')
    )
)

# --- 4. Visualizzazione finale ---

fig = go.Figure(data=[edge_trace, node_trace],
                layout=go.Layout(
                    title='<br>Grafo 3D (Circular) dell\'organizzazione terroristica, colorata secondo i Label',
                    showlegend=False,
                    height=1000,
                    margin=dict(l=0, r=0, b=0, t=50),
                    scene=dict(
                        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        zaxis=dict(showgrid=False, zeroline=False, showticklabels=False))))

fig.show()

In [ ]:
####--- 2D Spiral layout ---------###

pos = nx.spiral_layout(G)

# Estraiamo la lista dei valori dell'attributo 'label'
node_color_map = [G.nodes[node]['label'] for node in G.nodes()]

plt.figure(figsize=(20, 20))
nx.draw(
    G, 
    pos, 
    with_labels=False, 
    node_size=100, 
    node_color=node_color_map,  # Usa la lista appena creata
    cmap=plt.cm.coolwarm,       # Colormap per distinguere i gruppi 0 e 1
    font_size=5
)
plt.show()

In [ ]:
####--- 2D Random layout ---------###

pos = nx.random_layout(G)


# Estraiamo la lista dei valori dell'attributo 'label'
node_color_map = [G.nodes[node]['label'] for node in G.nodes()]

plt.figure(figsize=(20, 20))
nx.draw(
    G, 
    pos, 
    with_labels=False, 
    node_size=100, 
    node_color=node_color_map,  # Usa la lista appena creata
    cmap=plt.cm.coolwarm,       # Colormap per distinguere i gruppi 0 e 1
    font_size=5
)
plt.show()

In [ ]:
####--- 2D ARF layout ------------###

pos = nx.arf_layout(G)


# Estraiamo la lista dei valori dell'attributo 'label'
node_color_map = [G.nodes[node]['label'] for node in G.nodes()]

plt.figure(figsize=(20, 20))
nx.draw(
    G, 
    pos, 
    with_labels=False, 
    node_size=100, 
    node_color=node_color_map,  # Usa la lista appena creata
    cmap=plt.cm.coolwarm,       # Colormap per distinguere i gruppi 0 e 1
    font_size=5
)
plt.show()